In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Carregar o dataset Iris
iris = load_iris()
X = iris.data
y = iris.target

# Converter para DataFrame
df = pd.DataFrame(data=X, columns=iris.feature_names)
df['target'] = y

In [ ]:
# Particionamento dos dados
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Normalização dos dados
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Converter para tensores
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(4, 10)  # 4 features de entrada, 10 neurônios na primeira camada
        self.fc2 = nn.Linear(10, 3)   # 10 neurônios para 3 classes de saída (Iris Setosa, Versicolor, Virginica)
        self.relu = nn.ReLU()
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return self.softmax(x)

# Inicialização do modelo
model = MLP()

In [ ]:
# Definição de hiperparâmetros
num_epochs = 100
learning_rate = 0.01
batch_size = 16

# Definição da função de perda e otimizador
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Treinamento
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    # Backward pass e otimização
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

In [ ]:
# Avaliação no conjunto de validação
model.eval()
with torch.no_grad():
    val_outputs = model(X_val_tensor)
    _, predicted = torch.max(val_outputs, 1)

# Cálculo das métricas
print('\nClassification Report:')
print(classification_report(y_val_tensor, predicted))

# Matriz de confusão
conf_matrix = confusion_matrix(y_val_tensor, predicted)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

# Curva ROC e AUC
y_val_proba = torch.exp(val_outputs).numpy()  # Probabilidades
fpr, tpr, _ = roc_curve(y_val_tensor, y_val_proba[:, 1], pos_label=1)  # Para a classe '1'
roc_auc = roc_auc_score(y_val_tensor, y_val_proba, multi_class='ovr')

plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Salvando o modelo
torch.save(model.state_dict(), 'mlp_iris_model.pth')
print('Modelo salvo com sucesso!')